# Flask: From Basics to Intermediate (with Trading-Flavored Examples)

This notebook is a **from-scratch to intermediate** guide to **Flask**, one of the
most popular lightweight web frameworks for Python.

We’ll reuse the same **trading-flavored** study cases as in the FastAPI notebook
(symbols, prices, orders, positions), but implemented in Flask instead.

### Contents

- [1. What is Flask and When to Use It?](#1-what-is-flask-and-when-to-use-it)
- [2. Minimal App: Hello World and Simple Route](#2-minimal-app-hello-world-and-simple-route)
- [3. URL Path, Query Params, and JSON Bodies](#3-url-path-query-params-and-json-bodies)
- [4. Data Models with Pydantic (Optional but Helpful)](#4-data-models-with-pydantic-optional-but-helpful)
- [5. In-Memory Trading Endpoints (Symbols, Prices, Orders)](#5-in-memory-trading-endpoints-symbols-prices-orders)
- [6. Blueprints and Application Structure](#6-blueprints-and-application-structure)
- [7. Error Handling and Custom Responses](#7-error-handling-and-custom-responses)
- [8. Testing Flask Apps (FlaskClient / pytest)](#8-testing-flask-apps-flaskclient--pytest)
- [9. Async, Concurrency, and Background Work with Flask](#9-async-concurrency-and-background-work-with-flask)
- [10. Flask vs FastAPI vs Django](#10-flask-vs-fastapi-vs-django)

## 1. What is Flask and When to Use It?

Flask is a **minimal, flexible** WSGI web framework:

- Core is very small; most features come from **extensions** (Flask-SQLAlchemy,
  Flask-Login, etc.).
- Routes are just Python functions decorated with `@app.route`.
- Works well for **simple APIs**, dashboards, and services where you want to
  control which pieces you add.

In trading/quant contexts, Flask is still commonly used for:

- **Monitoring dashboards** and simple APIs.
- **Internal tools** and admin endpoints.
- Prototyping before moving to larger frameworks.

It’s synchronous by default; for **high-concurrency async I/O** you’d typically
use FastAPI / Starlette, but Flask is perfectly fine for a large class of tasks.

In [ ]:
# 1.1 Check Flask availability

try:
    import flask
    from flask import Flask
    print("Flask version:", flask.__version__)
except ImportError:
    print("Install Flask: pip install flask")

## 2. Minimal App: Hello World and Simple Route

A minimal Flask app in `app.py` looks like:

```python
from flask import Flask

app = Flask(__name__)

@app.route("/")
def hello():
    return {"message": "Hello World"}
```

Run with the dev server:

```bash
flask --app app run --reload
```

In this notebook, we’ll define `app` inline and use Flask’s built-in **test client**
to exercise endpoints without running an external server process.

In [ ]:
# 2.1 Minimal Flask app + test client demo

from flask import Flask, jsonify

app = Flask(__name__)


@app.route("/")
def root():
    # Returning a dict automatically becomes JSON in Flask >=1.1
    return {"message": "Hello from Flask Trading API"}


# Flask's test client lets us call routes without running a real server
with app.test_client() as client:
    resp = client.get("/")
    print("Status:", resp.status_code)
    print("JSON:", resp.get_json())

## 3. URL Path, Query Params, and JSON Bodies

Flask provides:

- **Path params** via `<name>` in the route string, e.g. `/symbols/<symbol>`.
- **Query params** via `request.args`.
- **JSON bodies** via `request.get_json()`.

We’ll implement:

- `GET /symbols` → optional `limit` query param.
- `GET /symbols/<symbol>` → path param.
- `GET /prices` → `symbol` query param.
- `POST /orders` → JSON body with order fields.

In [ ]:
# 3.1 Path and query parameters in Flask

from flask import request

FAKE_SYMBOLS = ["AAPL", "MSFT", "GOOG", "TSLA"]
FAKE_PRICES = {"AAPL": 180.0, "MSFT": 320.0, "GOOG": 140.0, "TSLA": 250.0}


@app.route("/symbols")
def list_symbols():
    # Query params are in request.args (MultiDict)
    try:
        limit = int(request.args.get("limit", "10"))
    except ValueError:
        limit = 10
    limit = max(1, min(limit, 100))
    return {"symbols": FAKE_SYMBOLS[:limit]}


@app.route("/symbols/<symbol>")
def get_symbol(symbol: str):
    available = symbol in FAKE_SYMBOLS
    return {"symbol": symbol, "available": available}


@app.route("/prices")
def get_price():
    symbol = request.args.get("symbol")
    if not symbol:
        return {"error": "symbol query param required"}, 400
    price = FAKE_PRICES.get(symbol)
    if price is None:
        return {"symbol": symbol, "error": "unknown symbol"}, 404
    return {"symbol": symbol, "price": price}


with app.test_client() as client:
    print("GET /symbols:", client.get("/symbols").get_json())
    print("GET /symbols/AAPL:", client.get("/symbols/AAPL").get_json())
    print("GET /prices?symbol=AAPL:", client.get("/prices?symbol=AAPL").get_json())

## 4. Data Models with Pydantic (Optional but Helpful)

Flask doesn’t enforce a particular way to define request/response schemas. You can:

- Manually parse `request.get_json()` and construct dicts.
- Or use **Pydantic** models (as in FastAPI) to validate and serialize.

Here we’ll use Pydantic to define `OrderIn` and `OrderOut` models, reusing the same
shapes as in the FastAPI notebook, but wiring them into Flask routes manually.

In [ ]:
# 4.1 Pydantic models for orders

from pydantic import BaseModel, Field
from enum import Enum


class Side(str, Enum):
    buy = "BUY"
    sell = "SELL"


class OrderType(str, Enum):
    market = "MARKET"
    limit = "LIMIT"


class OrderIn(BaseModel):
    symbol: str = Field(..., example="AAPL")
    side: Side
    type: OrderType
    quantity: float = Field(..., gt=0)
    limit_price: float | None = Field(None, description="Required for LIMIT orders")


class OrderOut(BaseModel):
    id: int
    symbol: str
    side: Side
    type: OrderType
    quantity: float
    limit_price: float | None
    status: str


ORDERS: dict[int, OrderOut] = {}
_next_order_id = 1


def create_order_internal(data: dict) -> tuple[dict, int]:
    """Core order creation logic shared between endpoints/tests.

    Returns (response_body, status_code).
    """
    global _next_order_id
    try:
        order_in = OrderIn(**data)
    except Exception as e:
        return {"error": str(e)}, 400

    if order_in.type is OrderType.limit and order_in.limit_price is None:
        return {"error": "limit_price required for LIMIT orders"}, 400

    oid = _next_order_id
    _next_order_id += 1
    order_out = OrderOut(
        id=oid,
        symbol=order_in.symbol,
        side=order_in.side,
        type=order_in.type,
        quantity=order_in.quantity,
        limit_price=order_in.limit_price,
        status="accepted",
    )
    ORDERS[oid] = order_out
    return order_out.dict(), 201


@app.route("/orders", methods=["POST"])
def create_order():
    payload = request.get_json(force=True, silent=True) or {}
    body, status = create_order_internal(payload)
    return body, status


@app.route("/orders/<int:order_id>")
def get_order(order_id: int):
    order = ORDERS.get(order_id)
    if not order:
        return {"error": "Order not found"}, 404
    return order.dict(), 200


with app.test_client() as client:
    resp = client.post(
        "/orders",
        json={
            "symbol": "AAPL",
            "side": "BUY",
            "type": "LIMIT",
            "quantity": 50,
            "limit_price": 180.0,
        },
    )
    print("POST /orders status:", resp.status_code)
    created = resp.get_json()
    print("Created order:", created)
    print("GET /orders/{id}:", client.get(f"/orders/{created['id']}").get_json())

## 5. In-Memory Trading Endpoints (Symbols, Prices, Orders)

We now have enough pieces to build a tiny **in-memory trading API** in Flask:

- `GET /symbols` and `GET /prices` for market data.
- `POST /orders` and `GET /orders/<id>` for order management.
- A `GET /positions/<symbol>` endpoint that reads from a fake position store.

This mirrors the FastAPI example but shows how you’d do it in a more manual Flask
style (explicit JSON handling, status codes, and validation).

In [ ]:
# 5.1 Simple in-memory positions endpoint

POSITIONS: dict[str, float] = {"AAPL": 100.0, "MSFT": -20.0}


@app.route("/positions/<symbol>")
def get_position(symbol: str):
    qty = POSITIONS.get(symbol)
    if qty is None:
        return {"symbol": symbol, "quantity": None, "error": "no position"}, 404
    return {"symbol": symbol, "quantity": qty}


with app.test_client() as client:
    print("GET /positions/AAPL:", client.get("/positions/AAPL").get_json())
    print("GET /positions/UNK:", client.get("/positions/UNK").status_code)

## 6. Blueprints and Application Structure

For anything beyond a toy app, you’ll want to split routes into **Blueprints** and
organize code into modules.

Example structure:

```text
app/
  __init__.py       # create_app factory, register blueprints
  routes/
    __init__.py
    symbols.py      # blueprint for /symbols
    orders.py       # blueprint for /orders
    positions.py    # blueprint for /positions
  models.py         # Pydantic models or dataclasses
  deps.py           # DB/session helpers
```

We’ll create a small `orders` blueprint and register it on the main app.

In [ ]:
# 6.1 Orders blueprint example

from flask import Blueprint

orders_bp = Blueprint("orders", __name__, url_prefix="/bp_orders")


@orders_bp.route("", methods=["GET"])
def list_orders_bp():
    # Reuse global ORDERS for demo
    return {"orders": [o.dict() for o in ORDERS.values()]}


@orders_bp.route("/<int:order_id>")
def get_order_bp(order_id: int):
    order = ORDERS.get(order_id)
    if not order:
        return {"error": "Order not found"}, 404
    return order.dict()


# Register blueprint on app
app.register_blueprint(orders_bp)


with app.test_client() as client:
    print("GET /bp_orders:", client.get("/bp_orders").get_json())

## 7. Error Handling and Custom Responses

Flask lets you customize error handling via:

- Returning `(body, status_code)` tuples from routes.
- Global **error handlers** registered with `app.errorhandler(ExceptionType)`.
- `abort(status_code, description=...)` for quick errors.

We’ll add a simple error handler that turns `ValueError` into a JSON 400 response.

In [ ]:
# 7.1 Global error handler for ValueError

@app.errorhandler(ValueError)
def handle_value_error(e: ValueError):
    # You can log here as well
    return {"error": str(e)}, 400


@app.route("/sqrt")
def sqrt_route():
    import math

    x_str = request.args.get("x")
    if x_str is None:
        raise ValueError("x query param required")
    try:
        x = float(x_str)
    except ValueError:
        raise ValueError("x must be a float")
    if x < 0:
        raise ValueError("x must be non-negative")
    return {"x": x, "sqrt": math.sqrt(x)}


with app.test_client() as client:
    print("GET /sqrt?x=9:", client.get("/sqrt?x=9").get_json())
    print("GET /sqrt:", client.get("/sqrt").status_code, client.get("/sqrt").get_json())

## 8. Testing Flask Apps (FlaskClient / pytest)

Testing Flask apps is straightforward with the built-in **test client** or
with pytest fixtures.

Basic pattern:

```python
from app import app

def test_root():
    with app.test_client() as client:
        r = client.get("/")
        assert r.status_code == 200
```

We’ll write a couple of small tests that exercise the order endpoints.
In a real project, move these into `tests/test_*.py` files and run with pytest.

In [ ]:
# 8.1 Example tests for orders

from flask.testing import FlaskClient


def test_create_and_get_order_flask(app: Flask):
    with app.test_client() as client_local:
        payload = {
            "symbol": "AAPL",
            "side": "BUY",
            "type": "MARKET",
            "quantity": 5,
        }
        r = client_local.post("/orders", json=payload)
        assert r.status_code == 201
        data = r.get_json()
        oid = data["id"]
        r2 = client_local.get(f"/orders/{oid}")
        assert r2.status_code == 200
        assert r2.get_json()["symbol"] == "AAPL"


# Run tests inline in notebook (using the global app)
if __name__ == "__main__":
    test_create_and_get_order_flask(app)
    print("Flask inline tests passed")
else:
    test_create_and_get_order_flask(app)
    print("Flask inline tests passed (notebook)")

## 9. Async, Concurrency, and Background Work with Flask

Classic Flask is **WSGI-based** and synchronous:

- Each request is handled in a worker (thread or process) depending on your WSGI server.
- You can run blocking I/O inside routes; concurrency comes from multiple workers.

Options for concurrency/background work:

- Run Flask behind **Gunicorn** with multiple workers/processes.
- Use a **task queue** (Celery, RQ) for heavy or slow background jobs.
- Use Flask’s `app.before_request` / `after_request` hooks for lightweight
  per-request background tasks.

Flask 2.x supports `async def` routes, but under a WSGI server they are effectively
run as synchronous coroutines. For truly async I/O (e.g. high-concurrency websockets),
prefer **FastAPI** / Starlette with an ASGI server.

## 10. Flask vs FastAPI vs Django

A brief, **practical comparison** for typical trading/quant services.

### 10.1 Flask

- **Philosophy**: microframework; you pick and assemble extensions.
- **Strengths**:
  - Very simple to start with; small core.
  - Huge ecosystem and many tutorials.
  - Good for small APIs, dashboards, internal tools.
- **Weaknesses**:
  - Synchronous by default; async support is limited by WSGI servers.
  - No built-in type-driven validation; you must add Pydantic/Marshmallow manually.

### 10.2 FastAPI

- **Philosophy**: modern, async-first API framework, type-driven.
- **Strengths**:
  - First-class **async** support (ASGI), great for high-concurrency I/O (REST, websockets).
  - Automatic **OpenAPI/Swagger UI** and docs.
  - Strong **type hints + Pydantic** → validation, clear contracts, better IDE support.
  - Very ergonomic for building medium-sized APIs quickly.
- **Weaknesses**:
  - More moving parts (ASGI, Pydantic versions, etc.).
  - Not as “batteries-included” as Django; you choose DB layer, auth, etc.

### 10.3 Django

- **Philosophy**: “batteries-included” full-stack web framework.
- **Strengths**:
  - Integrated ORM, admin, auth, templating, forms.
  - Great for traditional web apps, dashboards, and systems where you want
    one coherent stack.
- **Weaknesses**:
  - Historically sync/WSGI-oriented (though Django 3+ has async views support).
  - Heavier and more opinionated than Flask/FastAPI.

### 10.4 Choosing for trading/quant services

- **Flask**:
  - Good for **small, simple APIs**, internal tools, one-off dashboards.
  - You control exactly which extensions you pull in.
- **FastAPI**:
  - Great default for **new REST APIs** (order entry, risk, monitoring) that
    need good performance, validation, and async I/O.
  - Rich type hints and automatic docs help keep APIs maintainable.
- **Django**:
  - Use when you need a **full web app** (user accounts, admin, ORM-backed
    reporting dashboards) with lots of built-in functionality.

For many trading backends, a common pattern is:

- Use **FastAPI** (or Flask) for JSON APIs between services.
- Use **Django** or Flask for web-based dashboards and admin UIs.
- Keep ultra-low-latency order entry paths in C++/Rust or specialized gateways,
  exposing higher-level control/monitoring via these Python web frameworks.